In [407]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer


In [409]:
credits=pd.read_csv('tmdb_5000_credits.csv')
movies=pd.read_csv('tmdb_5000_movies.csv')

In [411]:
(credits.shape),(movies.shape)

((4803, 4), (4803, 20))

In [413]:
movies=movies.merge(credits,on='title')

In [415]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='object')

In [417]:
movies=movies[['movie_id','genres','title','cast','crew','keywords','overview']]

In [419]:
movies.isnull().sum()

movie_id    0
genres      0
title       0
cast        0
crew        0
keywords    0
overview    3
dtype: int64

In [421]:
movies.dropna(inplace=True)

In [423]:
movies.duplicated().sum()

0

In [425]:
movies.iloc[0]['genres']

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [427]:
import ast

def convert(title_obj):
    list=[]
    for i in ast.literal_eval(title_obj):
        list.append(i['name'])
    return list

In [429]:
movies['genres']=movies['genres'].apply(convert)

In [431]:
movies['keywords']=movies['keywords'].apply(convert)

In [433]:
def convert3(title_obj):
    list=[]
    counter=0
    for i in ast.literal_eval(title_obj):
        if counter !=3:
            list.append(i['name'])
            counter+=1
        else:
            break
    return list

In [435]:
movies['cast']=movies['cast'].apply(convert3)

In [437]:
def directors(obj):
    drct_list=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            drct_list.append(i['name'])
            break
    return drct_list
            

In [439]:
movies['crew']=movies['crew'].apply(directors)

In [441]:
movies['genres']=movies['genres'].astype(str).str.replace(' ','')
movies['cast']=movies['cast'].astype(str).str.replace(' ','')
movies['crew']=movies['crew'].astype(str).str.replace(' ','')
movies['keywords']=movies['keywords'].astype(str).str.replace(' ','')

In [443]:
movies['overview']=movies['overview'].apply(lambda x: x.split())
movies['genres']=movies['genres'].apply(lambda x: x.split())
movies['cast']=movies['cast'].apply(lambda x: x.split())
movies['crew']=movies['crew'].apply(lambda x: x.split())
movies['keywords']=movies['keywords'].apply(lambda x: x.split())

In [445]:
movies['tags']=movies['overview']+movies['genres']+movies['cast']+movies['crew']+movies['keywords']

In [447]:
print(type(movies['overview'][0]))
print(type(movies['genres'][0]))
print(type(movies['cast'][0]))
print(type(movies['crew'][0]))
print(type(movies['keywords'][0]))

<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>


In [449]:
movies=movies.drop(columns=['genres','cast','crew','keywords','overview'])

In [451]:
movies['tags']=movies['tags'].apply(lambda x: ' '.join(x))

In [453]:
movies['tags']=movies['tags'].apply(lambda x: x.lower())

In [455]:
cv=CountVectorizer(stop_words='english',max_features=5000)
vectors=cv.fit_transform(movies['tags']).toarray()

In [ ]:
for items in cv.get_feature_names_out():
    print(items)

In [463]:
vectors[0]


array([0, 0, 0, ..., 0, 0, 0])

In [465]:
import nltk
from nltk.stem.porter import PorterStemmer 
ps=PorterStemmer()

def stem(text_items):
    list_items=[]
    
    for i in text_items.split():
        list_items.append(ps.stem(i))
    
    return ' '.join(list_items)

In [467]:
(ps.stem('actions')),(ps.stem('action'))
(ps.stem('actors')),(ps.stem('actor'))

('actor', 'actor')

In [469]:
stem(movies['tags'][0])

"in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. ['action','adventure','fantasy','sciencefiction'] ['samworthington','zoesaldana','sigourneyweaver'] ['jamescameron'] ['cultureclash','future','spacewar','spacecolony','society','spacetravel','futuristic','romance','space','alien','tribe','alienplanet','cgi','marine','soldier','battle','loveaffair','antiwar','powerrelations','mindandsoul','3d']"

In [471]:
movies['tags']=movies['tags'].apply(stem)

In [473]:
movies['tags'][1]

'captain barbossa, long believ to be dead, ha come back to life and is head to the edg of the earth with will turner and elizabeth swann. but noth is quit as it seems. [\'adventure\',\'fantasy\',\'action\'] [\'johnnydepp\',\'orlandobloom\',\'keiraknightley\'] [\'goreverbinski\'] [\'ocean\',\'drugabuse\',\'exoticisland\',\'eastindiatradingcompany\',"loveofone\'slife",\'traitor\',\'shipwreck\',\'strongwoman\',\'ship\',\'alliance\',\'calypso\',\'afterlife\',\'fighter\',\'pirate\',\'swashbuckler\',\'aftercreditsstinger\']'

In [475]:
cv=CountVectorizer(stop_words='english',max_features=5000)
vectors=cv.fit_transform(movies['tags']).toarray()

In [477]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0])

In [ ]:
for i in cv.get_feature_names_out():
    print(i)

In [481]:
# Calculate Cosine Distance 

from sklearn.metrics.pairwise import cosine_similarity

In [483]:
similarity=cosine_similarity(vectors)

In [484]:
similarity[0].shape

(4806,)

In [487]:
movies[movies['title']=='The Dark Knight Rises'].index[0]

3

In [489]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x: x[1])[1:6]

[(1214, 0.2622900470777891),
 (507, 0.25903973506580724),
 (539, 0.2537477434955704),
 (2405, 0.2536731447159205),
 (1202, 0.24507154069793594)]

In [491]:
def recommended(movies_text):
    get_index=movies[movies['title']==movies_text].index[0]
    get_distance=similarity[get_index]
    recommnd_list=sorted(list(enumerate(get_distance)),reverse=True,key=lambda x: x[1])[1:6]
    
    for i in recommnd_list:
        print(movies.iloc[i[0]].title)
    

In [493]:
recommended('Superman')

Superman Returns
Superman II
Superman III
Iron Man 2
Ant-Man


In [495]:
import pickle

pickle.dump(movies,open('movies.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))